# EMAS - Evaluation Notebook (Colab)

This notebook is thought to be run cell by cell, in order to obtain our proposed result, each time varying the evaluation dataset and used method.

### Install Dependencies and Libraries

The following cell installs all the libraries needed to make the models and evaluation metrics work.

In [ ]:
!pip install lightning
!pip install torch torchvision
!pip install "opencv-python<4.10" 
!pip install scikit-learn matplotlib visdom pillow
!pip install ood-metrics
!pip install -q timm peft lightning transformers

### Setup

We decided to work with Google Drive. Thus, by running the following code, you'll mount the drive and copy the Cityscapes dataset from the drive storage to the local disk of Google Colab, unzipping to grant optimal performance during evaluation process.

NOTE: you need to pay attention to datasets paths in case you mounted your own drive differently or you have a different directory organization.

In [ ]:
import os
import shutil
import time
from google.colab import drive

# 1. Drive Mounting
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. PATH CONFIGURATION
DRIVE_CITY_DIR = "/content/drive/MyDrive/Anomaly_Segmentation/Cityscapes"
LOCAL_CITY_DIR = "/content/Cityscapes_Local"

def copy_and_unzip(zip_name, drive_dir, local_dest):
    source = os.path.join(drive_dir, zip_name)
    if not os.path.exists(source):
        print(f" ERROR: Can't find {source} in the Drive!")
        return
    os.makedirs(local_dest, exist_ok=True)
    print(f"Copy and unzip of {zip_name}...")
    !unzip -q "{source}" -d "{local_dest}"
    print(f" Completed for {zip_name}")

# 3. Setup Cityscapes
copy_and_unzip("leftImg8bit_trainvaltest.zip", DRIVE_CITY_DIR, LOCAL_CITY_DIR)
copy_and_unzip("gtFine_trainvaltest.zip", DRIVE_CITY_DIR, LOCAL_CITY_DIR)

print("\n DATASET SETUP COMPLETED!")

### mIoU Evaluation

To evaluate the segmentation accuracy we proceed by calculating the mIoU on Cityscapes validation dataset.

NOTE: you need to change the path according to the configuration you are evaluating, if No-Energy, EMAS or EMAS-HN. The cell below is already prepared to run for EMAS-HN configuration.

In [ ]:
!python eval_miou.py \
  --ckpt "/content/drive/MyDrive/Anomaly_Segmentation/MaskArchitectureAnomaly_CourseProject/EMAS/checkpoints_last_ramp/bestOOD-epoch=34-step=014000-new.ckpt" \
  --city_root "/content/Cityscapes_Local" \
  --split val \
  --img_h 1024 --img_w 1024 \
  --batch_size 1 \
  --num_workers 2

### Evaluation

Now we pass to Anomaly Detection evaluation, using different methods on different dataset.

To obtain our proposed results, remember to change each time the evaluation dataset (RoadAnomaly21, RoadObsticle21, FS_LostFound_full, fs_static, RoadAnomaly) respectively changing also the image format (*.png, *.webp, *.png, *.jpg, *.jpg).

The available methods are msp, maxlogit, maxentropy, rba, to correctly change in parameter --method.

NOTE: you need to change the path according to the configuration you are evaluating, if No-Energy, EMAS or EMAS-HN. The cell below is already prepared to run for EMAS-HN configuration.

In [ ]:
# Example on RoadAnomaly21
!python eval.py \
  --input "/content/drive/MyDrive/Anomaly_Segmentation/RoadAnomaly21/images/*.png" \
  --ckpt_dir  "/content/drive/MyDrive/Anomaly_Segmentation/MaskArchitectureAnomaly_CourseProject/EMAS/checkpoints_last_ramp" \
  --ckpt_name "bestOOD-epoch=34-step=014000-new.ckpt" \
  --img_h 1024 --img_w 1024 \
  --method msp

### Temperature Scaling Evaluation

The following cell is used to evaluate the best Temperature T parameter to improve anomaly separation from normal samples.

To obtain our proposed results, remember to change each time the evaluation dataset (RoadAnomaly21, RoadObsticle21, FS_LostFound_full, fs_static, RoadAnomaly) respectively changing also the image format (*.png, *.webp, *.png, *.jpg, *.jpg).

NOTE: you need to change the path according to the configuration you are evaluating, if No-Energy, EMAS or EMAS-HN. The cell below is already prepared to run for EMAS-HN configuration.

In [ ]:
import os

INPUT_IMAGES = "/content/drive/MyDrive/Anomaly_Segmentation/RoadAnomaly/images/*.jpg"
LOAD_DIR = "/content/drive/MyDrive/Anomaly_Segmentation/MaskArchitectureAnomaly_CourseProject/EMAS/checkpoints_last_ramp"
LOAD_WEIGHTS = "bestOOD-epoch=34-step=014000-new.ckpt"

cmd = f"python evalTemp.py --input '{INPUT_IMAGES}' --loadDir '{LOAD_DIR}' --loadWeights '{LOAD_WEIGHTS}'"
print(f"Esecuzione: {cmd}")
!{cmd}